# Inicialización del Backend de Detección de Sátira en Google Colab

Configuración del entorno del servidor backend de Django en Google Colab. Instala las dependencias necesarias con versiones compatibles de transformers, enlaza los archivos del modelo y expone el servicio a la red externa usando un túnel seguro.

## 1. Conexión a Google Drive
Montaje de la unidad de Google Drive para la lectura de los pesos y serializadores del modelo clasificador.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Descarga del código fuente
Clonación del repositorio Git de la aplicación y navegación al directorio del proyecto.

In [ ]:
import os
if not os.path.exists('/content/DeteccionSatira'):
    print("Clonando el repositorio de GitHub...")
    !git clone https://github.com/luisknight24/DeteccionSatira.git /content/DeteccionSatira
else:
    print("El repositorio ya existe. Actualizando cambios...")
    %cd /content/DeteccionSatira
    !git pull

%cd /content/DeteccionSatira

## 3. Instalación de requerimientos
Instalación de los paquetes requeridos y de la versión específica de transformers para prevenir inconsistencias de deserialización.

In [ ]:
!pip install -r requirements.txt
!pip install transformers==4.44.2
!python -m spacy download es_core_news_sm
!python satire_detector_api/initialize_nltk.py

## 4. Enlace de recursos y modelos
Copia de los archivos de pesos del modelo, vectorizadores TF-IDF, scalers y archivos del tokenizador al directorio de archivos estáticos de Django.

In [ ]:
# Creación de la estructura de directorios requerida
!mkdir -p satire_detector_api/static/model_files

import os

# Rutas de origen definidas para la extracción de modelos
ruta_drive = "/content/drive/MyDrive/Titulacion/Modelos"
ruta_git_model = "documentos_origen/Titulacion1/Entrenamiento/Transfomer/bert-base-spanish"

if os.path.exists(os.path.join(ruta_drive, "best_model_spanish_loss.pt")):
    print("📂 Cargando modelos desde la carpeta de Drive...")
    !cp "{ruta_drive}/best_model_spanish_loss.pt" satire_detector_api/static/
    !cp "{ruta_drive}/tfidf_vectorizer.pkl" satire_detector_api/static/
    !cp "{ruta_drive}/minmax_scaler.pkl" satire_detector_api/static/
    !cp -r "{ruta_drive}/tokenizer_files" satire_detector_api/static/model_files/
    print("✅ Modelos enlazados con éxito desde Google Drive.")
elif os.path.exists(os.path.join(ruta_git_model, "best_model_spanish_loss.pt")):
    print("📂 Cargando modelos desde la copia local del repositorio...")
    !cp {ruta_git_model}/best_model_spanish_loss.pt satire_detector_api/static/
    !cp {ruta_git_model}/caracteristicas/tfidf_vectorizer.pkl satire_detector_api/static/
    !cp {ruta_git_model}/caracteristicas/minmax_scaler.pkl satire_detector_api/static/
    !cp -r {ruta_git_model}/caracteristicas/tokenizer_files satire_detector_api/static/model_files/
    print("✅ Modelos enlazados con éxito desde los archivos locales.")
else:
    print("❌ ERROR: No se pudieron localizar los archivos necesarios del clasificador.")
    print(f"Verificar que los archivos requeridos estén presentes en Drive bajo la ruta: {ruta_drive}")

## 5. Ejecución del backend y apertura del túnel
Procedimiento de arranque del servidor Django en segundo plano y exposición de la API a internet mediante un túnel SSH estable con detección automática de fallos de red.

In [ ]:
import subprocess
import time
import os
import fcntl

# Limpieza de procesos huérfanos de ejecuciones anteriores para liberar el puerto 8000
print("🧹 Limpiando procesos de Django y túneles en ejecución...")
!pkill -f "manage.py runserver"
!pkill -f "ssh -o StrictHostKeyChecking"
!pkill -f "lt --port"
!pkill -f "localtunnel"
time.sleep(3)

print("🚀 Inicializando Django en puerto 8000...")
proc_django = subprocess.Popen(
    ["python", "-u", "satire_detector_api/manage.py", "runserver", "127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Configurar el stream de salida como no-bloqueante para leer en tiempo real
fd = proc_django.stdout.fileno()
fl = fcntl.fcntl(fd, fcntl.F_GETFL)
fcntl.fcntl(fd, fcntl.F_SETFL, fl | os.O_NONBLOCK)

# Esperar a que Django responda activamente en el puerto 8000
print("⏳ Esperando a que el servidor de Django cargue los modelos y comience a escuchar...")
print("(Esto puede tomar de 20 a 50 segundos debido al montaje de red de Google Drive)")
print("--- LOGS DE ARRANQUE DE DJANGO EN TIEMPO REAL ---")
django_listo = False

for i in range(75):
    time.sleep(1)
    
    # Intentar leer y mostrar los logs acumulados sin bloquear la ejecución
    try:
        logs = proc_django.stdout.read()
        if logs:
            print(logs, end="", flush=True)
    except Exception:
        pass
        
    # Verificar si el proceso murió
    if proc_django.poll() is not None:
        print(f"\n❌ Django se detuvo inesperadamente con código de salida: {proc_django.returncode}")
        try:
            logs_finales = proc_django.stdout.read()
            if logs_finales:
                print(logs_finales, end="", flush=True)
        except Exception:
            pass
        django_listo = False
        break
    
    # Probar si responde el puerto local mediante curl cada 2 segundos
    if i % 2 == 0:
        resultado_curl = subprocess.run(
            ["curl", "-s", "-o", "/dev/null", "-w", "%{http_code}", "http://127.0.0.1:8000/api/"],
            capture_output=True,
            text=True
        )
        codigo_http = resultado_curl.stdout.strip()
        if codigo_http in ["200", "404", "405"]:
            print(f"\n✅ ¡Django está listo y respondiendo! (Código HTTP {codigo_http})")
            django_listo = True
            break

if django_listo:
    print("\n🌐 Iniciando túnel SSH seguro...")
    print("Intentando conectar primero con pinggy (pgy.in)...")
    
    proc_tunnel = subprocess.Popen(
        ["ssh", "-o", "StrictHostKeyChecking=no", "-R", "80:127.0.0.1:8000", "pgy.in"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    
    # Esperar 4 segundos para comprobar si la resolución DNS o la conexión falló
    time.sleep(4)
    if proc_tunnel.poll() is not None:
        salida_error = proc_tunnel.stdout.read().strip()
        print(f"⚠️ Conexión con pinggy fallida o sin resolución DNS: {salida_error}")
        print("🔄 Cambiando automáticamente a serveo.net (fallback)...\n")
        
        # Abrir serveo.net en primer plano directamente
        print("🌐 Iniciando túnel SSH seguro con serveo.net...")
        print("(Copia la URL '.serveo.net' e ingrésala en tu Frontend de Angular)")
        !ssh -o StrictHostKeyChecking=no -R 80:127.0.0.1:8000 serveo.net
    else:
        print("✅ Conexión con pinggy establecida.")
        print("(Copia la URL '.pinggy.link' e ingrésala en tu Frontend de Angular)\n")
        
        # Hacer no-bloqueante la salida de pinggy para mostrar la URL
        fd_t = proc_tunnel.stdout.fileno()
        fl_t = fcntl.fcntl(fd_t, fcntl.F_GETFL)
        fcntl.fcntl(fd_t, fcntl.F_SETFL, fl_t | os.O_NONBLOCK)
        
        try:
            while True:
                time.sleep(1)
                logs_t = proc_tunnel.stdout.read()
                if logs_t:
                    print(logs_t, end="", flush=True)
                if proc_tunnel.poll() is not None:
                    print("\n❌ El túnel de pinggy se desconectó.")
                    break
        except KeyboardInterrupt:
            print("\n⏹️ Deteniendo túnel...")
            proc_tunnel.terminate()
else:
    print("\n❌ No se pudo iniciar el túnel porque el servidor Django no respondió a tiempo.")